# 02 — Baselines: OCR+правила vs OCR+правила+LLM

**Задача:** замерить и сравнить качество извлечения полей в двух режимах.

| Условное имя | Что в пайплайне                                              | Когда удобен                      |
|--------------|--------------------------------------------------------------|-----------------------------------|
| **Baseline** | Tesseract OCR + regex/правила (без LLM)                       | CPU-only, требует скорости, офлайн |
| **Improved** | Tesseract OCR + regex/правила + локальная мультимодальная LLM (Ollama, `llama3.2-vision`) поверх | Есть GPU/CPU-бюджет, нужна точность на сложных документах |

Метрика — **поле-level accuracy** на контрольной выборке `samples/control_samples.json`: для каждого примера сравниваем только те поля, у которых задан непустой `expected`. Так оценка отражает качество там, где есть истина, а не «глобальную точность» на произвольных документах.

In [ ]:
import json
import sys
import time
from pathlib import Path
from collections import defaultdict

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

SAMPLES_FILE = PROJECT_ROOT / 'samples' / 'control_samples.json'
DOCS_DIR = PROJECT_ROOT / 'приложения'

with SAMPLES_FILE.open('r', encoding='utf-8') as f:
    control = json.load(f)

samples = control['samples']
fields = control['fields']
print(f'Контрольных примеров: {len(samples)}')
print(f'Поля: {fields}')

## 1. Подгружаем пайплайн

In [ ]:
from src.service import extract_document_payload
from src.eval import compare_eval_value

def run_one(file_path: Path, allow_llm: bool) -> dict:
    """Прогон одного файла через extract_document_payload в выбранном режиме."""
    fbytes = file_path.read_bytes()
    return extract_document_payload(
        fbytes,
        file_path.name,
        allow_llm=allow_llm,
        scan_release_date=False,  # быстрый режим, как в /api/evaluate/default?fast=1
    )

def score_sample(expected: dict, predicted: dict) -> dict:
    """Сравнение по непустым полям expected, возвращает словарь правильных/всего."""
    correct = 0
    total = 0
    per_field = {}
    for field, exp_value in expected.items():
        if exp_value in (None, '', [], {}):
            continue
        total += 1
        pred_value = predicted.get(field)
        ok = compare_eval_value(exp_value, pred_value)
        per_field[field] = ok
        if ok:
            correct += 1
    return {'correct': correct, 'total': total, 'per_field': per_field}

## 2. Запуск baseline (OCR + правила, без LLM)

In [ ]:
baseline_results = []
t0 = time.perf_counter()
for s in samples:
    fp = DOCS_DIR / s['filename']
    if not fp.exists():
        print(f'!! пропускаю (нет файла): {s["filename"]}')
        continue
    pred = run_one(fp, allow_llm=False)
    score = score_sample(s['expected'], pred)
    baseline_results.append({'file': s['filename'], **score})
    print(f"{s['filename'][:60]:<60} {score['correct']}/{score['total']}")
baseline_elapsed = time.perf_counter() - t0
print(f'\nВремя baseline: {baseline_elapsed:.1f} сек')

## 3. Запуск improved (с LLM)

> Требует запущенной Ollama: `ENABLE_LLM=1` и доступная модель (`OLLAMA_MODEL`). Если Ollama недоступна, ячейка просто упадёт в OCR-only — это штатное поведение fallback'а, см. README.

In [ ]:
improved_results = []
t0 = time.perf_counter()
for s in samples:
    fp = DOCS_DIR / s['filename']
    if not fp.exists():
        continue
    pred = run_one(fp, allow_llm=True)
    score = score_sample(s['expected'], pred)
    improved_results.append({'file': s['filename'], **score})
    print(f"{s['filename'][:60]:<60} {score['correct']}/{score['total']}")
improved_elapsed = time.perf_counter() - t0
print(f'\nВремя improved: {improved_elapsed:.1f} сек')

## 4. Сводка

In [ ]:
def aggregate(results):
    correct = sum(r['correct'] for r in results)
    total = sum(r['total'] for r in results)
    accuracy = (correct / total) if total else 0.0
    return correct, total, accuracy

b_correct, b_total, b_acc = aggregate(baseline_results)
i_correct, i_total, i_acc = aggregate(improved_results)

print('Модель    | Правильно/Всего | Accuracy | Время (сек)')
print('----------+-----------------+----------+------------')
print(f'Baseline  | {b_correct:>6}/{b_total:<7} | {b_acc:.3f}    | {baseline_elapsed:.1f}')
print(f'Improved  | {i_correct:>6}/{i_total:<7} | {i_acc:.3f}    | {improved_elapsed:.1f}')

## 5. Разбор ошибок по полям

In [ ]:
def per_field_table(results, label):
    by_field = defaultdict(lambda: [0, 0])  # correct, total
    for r in results:
        for f, ok in r['per_field'].items():
            by_field[f][1] += 1
            if ok:
                by_field[f][0] += 1
    print(f'\n[{label}] точность по полям:')
    for f, (c, t) in sorted(by_field.items()):
        acc = c / t if t else 0
        print(f'  {f:<22} {c}/{t}  ({acc:.2f})')

per_field_table(baseline_results, 'Baseline')
per_field_table(improved_results, 'Improved')

## 6. Выводы и выбор финальной модели

Реальные цифры из прогона через `GET /api/evaluate/default` в Docker (6 PDF, 20 непустых проверок полей, Ollama с `llama3.2-vision:latest`):

| Режим                            | Поля верно / всего | Accuracy | Время  |
|----------------------------------|--------------------|----------|--------|
| Baseline (OCR+правила)           | 20 / 20            | **100 %** | 26.0 с |
| Improved (+ LLM доступна, full)  | 20 / 20            | **100 %** | 32.2 с |

**Ключевое наблюдение:** в режиме Improved LLM не вызвалась ни на одном документе:

- 3 документа → `"LLM skipped: OCR+rules confidence is already high"` (baseline уже даёт высокий `quality_score`).
- 3 документа → `"LLM skipped: likely non-passport/service input"` (`_should_skip_llm_for_file` отсёк по содержимому/имени — это разбор паспорта / экранная форма).

Разница 6 секунд между прогонами — это не LLM (она не вызывалась), а доскан-поиск даты выпуска по штампам (`scan_release_date=True`).

По полям все 100 %: `document_type` 6/6, `kod_dokumenta` 4/4, `naimenovanie` 4/4, `proizvoditel` 3/3, `data_vypuska` 1/1, `kod_zakaza` 1/1, `zavodskie_nomera` 1/1.

**Финальный выбор для prod-сервиса — гибрид** (`_extract_document_payload` в [`app/app.py`](../app/app.py)):

1. Сначала baseline (OCR+правила) — быстро, оффлайн, предсказуемо.
2. Если baseline даёт низкий `quality_score` или неполный набор полей **и** документ похож на паспорт **и** LLM доступна → доскан LLM.
3. При любой проблеме с LLM → silent fallback на baseline.

На нашем корпусе шаг 2 не сработал ни разу — baseline уже идеален. Это лучше, чем «всегда LLM»: меньше нагрузка на инфраструктуру, латентность секунды вместо десятков секунд, нет жёсткой зависимости от доступности Ollama в моменте.

**Ограничение:** корпус маленький (7 PDF в `приложения/`, 6 с проверками в контроле). На более широкой выборке с плохими сканами и нестандартными вёрстками часть полей неизбежно потеряется, и LLM-этап начнёт реально включаться. Расширение контрольной выборки — в планах (см. `report.md` §8).